# Green NLP Benchmark Lab — Notebook 3c: Three Levers, One Run, Closed Book — coding workshop (advanced)

**Content warning:** Part 2 uses sentences from *Speechwriter* (Paulo Naseiro), including
two with the novel's own coarse language (English "fuck"/"shit"-level swearing). They are
kept unfiltered because idiomatic profanity is itself a translation challenge.

Notebooks 1–3 measured one configuration per session. This notebook runs all three in a
single Colab session, on the same 26 questions from Notebook 3:

| Configuration | Model | What the model reads |
|---|---|---|
| **Baseline** | Qwen2.5-1.5B-Instruct, fp16 | the question only |
| **Quantized** | Qwen2.5-1.5B-Instruct, GPTQ int4 | the question only |
| **Retrieval (RAG)** | Qwen2.5-1.5B-Instruct, fp16 | the question + the 2 retrieved documents |

The baseline and quantized runs are **closed book**: no documents in the prompt, so the
model answers from what it learned in training. Notebooks 2 and 3 handed the model
documents every time, which turns the quiz into reading comprehension. Here documents
reach the model only through retrieval. The comparison therefore shows what quantization
does to what the model knows, and what retrieval adds to it.

Part 2 runs the translation thread the same way: fp16, quantized, and fp16 with a
retrieved translation-memory example, evaluated with the four metrics from earlier
sessions.

**Before you start:** `Runtime` → `Change runtime type` → **T4 GPU** → `Save`. Nothing needs
uploading: this notebook writes its own result files.

Expected time: 70–90 minutes, including 5–10 minutes of installation.

## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried.
   Or click **Fill in solution** right above the exercise cell, then run the exercise cell:
   the solution replaces the code in it and runs. **Fill in solutions**, under these
   instructions, does this for every exercise.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

In [ ]:
# @title Run this cell first: it sets up the "Fill in solution" buttons (point 3 above).
import re

from IPython.display import HTML, JSON, display

_solutions = {
    1: "BASE_MODEL_NAME = \"Qwen/Qwen2.5-1.5B-Instruct\"\nGPTQ_MODEL_NAME = \"Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4\"\n\ntokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)\nload_info = {}\n\n\ndef load_timed(label, loader):\n    mem_before = gpu_memory_gb()\n    start = time.time()\n    loaded = loader()\n    loaded.eval()\n    loaded.generation_config.do_sample = False\n    loaded.generation_config.temperature = None\n    loaded.generation_config.top_p = None\n    loaded.generation_config.top_k = None\n    mem_after = gpu_memory_gb()\n    load_info[label] = {\n        \"load_seconds\": round(time.time() - start, 1),\n        \"weights_gb\": None if mem_after is None else round(mem_after - mem_before, 2),\n    }\n    return loaded\n\n\nmodel_fp16 = load_timed(\"fp16\", lambda: AutoModelForCausalLM.from_pretrained(\n    BASE_MODEL_NAME, dtype=torch.float16, device_map=\"auto\"))\n\ntry:\n    model_quant = load_timed(\"quantized\", lambda: AutoModelForCausalLM.from_pretrained(\n        GPTQ_MODEL_NAME, device_map=\"auto\", quantization_config=GPTQConfig(bits=4, backend=\"torch\")))\n    QUANT_CONFIG, QUANT_MODEL_NAME = \"gptq-int4\", GPTQ_MODEL_NAME\nexcept Exception as e:\n    print(f\"GPTQ load failed ({type(e).__name__}: {e}). Falling back to bitsandbytes int8.\")\n    model_quant = load_timed(\"quantized\", lambda: AutoModelForCausalLM.from_pretrained(\n        BASE_MODEL_NAME, quantization_config=BitsAndBytesConfig(load_in_8bit=True),\n        dtype=torch.float16, device_map=\"auto\"))\n    QUANT_CONFIG, QUANT_MODEL_NAME = \"bnb-int8\", BASE_MODEL_NAME\n\nMODELS = {\"fp16\": model_fp16, \"quantized\": model_quant}\nprint(f\"Quantized path: {QUANT_CONFIG}\")\npd.DataFrame(load_info).T",
    2: "def build_prompt(question, doc_names):\n    if not doc_names:\n        messages = [\n            {\"role\": \"system\",\n             \"content\": \"You are a concise assistant. Answer the question in one short sentence.\"},\n            {\"role\": \"user\", \"content\": f\"Question: {question}\"},\n        ]\n    else:\n        context = \"\\n\\n\".join(f\"[{name}]\\n{CORPUS[name].strip()}\" for name in doc_names)\n        messages = [\n            {\"role\": \"system\",\n             \"content\": (\"You are a concise assistant. Answer the question using ONLY the \"\n                         \"documents provided below. Answer in one short sentence.\")},\n            {\"role\": \"user\", \"content\": f\"Documents:\\n{context}\\n\\nQuestion: {question}\"},\n        ]\n    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n\ndef answer_question(model, question, doc_names, max_new_tokens=60):\n    inputs = tokenizer(build_prompt(question, doc_names), return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)\n    n_prompt = inputs[\"input_ids\"].shape[1]\n    return tokenizer.decode(output[0][n_prompt:], skip_special_tokens=True).strip(), n_prompt\n\n\ndef score_answer(answer, reference_list):\n    return any(ref.lower() in answer.lower() for ref in reference_list)\n\n\nCONFIGS = {\n    \"baseline-fp16-closed-book\": {\"model\": \"fp16\", \"retrieval\": False},\n    f\"{QUANT_CONFIG}-closed-book\": {\"model\": \"quantized\", \"retrieval\": False},\n    f\"retrieval-top{TOP_K}-fp16\": {\"model\": \"fp16\", \"retrieval\": True},\n}\n\nq = QA_PAIRS[0][\"question\"]\nprint(\"Q:\", q)\nfor config, spec in CONFIGS.items():\n    docs = retrieve(q) if spec[\"retrieval\"] else []\n    print(f\"  {config:28s} {answer_question(MODELS[spec['model']], q, docs)[0]}\")",
    3: "runs = {}\nfor config, spec in CONFIGS.items():\n    model = MODELS[spec[\"model\"]]\n    tracker = make_tracker(f\"green-3c-{config}\")\n    rows = []\n    tracker.start()\n    start_time = time.time()\n    for pair in QA_PAIRS:\n        docs = retrieve(pair[\"question\"]) if spec[\"retrieval\"] else []\n        answer, n_prompt = answer_question(model, pair[\"question\"], docs)\n        rows.append({**pair, \"docs\": docs, \"answer\": answer, \"prompt_tokens\": n_prompt,\n                     \"correct\": score_answer(answer, pair[\"reference\"]),\n                     \"retrieval_correct\": pair[\"source\"] in docs if spec[\"retrieval\"] else None})\n    elapsed = time.time() - start_time\n    emissions_kg = tracker.stop()\n    runs[config] = {\"rows\": rows, \"elapsed\": elapsed, \"emissions_kg\": emissions_kg,\n                    \"energy_kwh\": energy_kwh_of(tracker), **spec}\n    print(f\"{config:28s} {elapsed:6.1f} s  {emissions_kg * 1000:.4f} g CO2eq\")",
    4: "config_names = list(runs)\nfor i, pair in enumerate(QA_PAIRS):\n    print(f\"Q: {pair['question']}\")\n    for config in config_names:\n        r = runs[config][\"rows\"][i]\n        print(f\"   {config:28s} {'✓' if r['correct'] else '✗'}  {r['answer']}\")\n    print()\n\nsummary = pd.DataFrame({\n    config: {\n        \"answer accuracy\": np.mean([r[\"correct\"] for r in run[\"rows\"]]),\n        \"accuracy, Notebook 1's 8 questions\": np.mean(\n            [r[\"correct\"] for r in run[\"rows\"][:N_SESSION1_QUESTIONS]]),\n        \"retrieval accuracy\": (np.mean([r[\"retrieval_correct\"] for r in run[\"rows\"]])\n                               if run[\"retrieval\"] else None),\n        \"mean prompt tokens\": np.mean([r[\"prompt_tokens\"] for r in run[\"rows\"]]),\n        \"seconds per question\": run[\"elapsed\"] / len(QA_PAIRS),\n        \"g CO2eq per question\": run[\"emissions_kg\"] * 1000 / len(QA_PAIRS),\n        \"model weights (GB)\": load_info[run[\"model\"]][\"weights_gb\"],\n    }\n    for config, run in runs.items()\n})\nsummary",
    5: "def joules(kwh):\n    return kwh * 3.6e6\n\n\ndef joules_per_correct(kwh, rows):\n    n_correct = sum(r[\"correct\"] for r in rows)\n    return joules(kwh) / n_correct if n_correct else float(\"inf\")\n\nrows = []\nfor config, run in runs.items():\n    n_correct = sum(r[\"correct\"] for r in run[\"rows\"])\n    rows.append({\"config\": config,\n                 \"accuracy\": n_correct / len(run[\"rows\"]),\n                 \"J per question\": joules(run[\"energy_kwh\"]) / len(run[\"rows\"]),\n                 \"J per correct answer\": joules_per_correct(run[\"energy_kwh\"], run[\"rows\"])})\nefficiency = pd.DataFrame(rows)\nefficiency",
    6: "def dominates(a, b, cost, quality):\n    \"\"\"True if row a dominates row b: no worse on both, strictly better on at least one.\"\"\"\n    no_worse = a[cost] <= b[cost] and a[quality] >= b[quality]\n    better = a[cost] < b[cost] or a[quality] > b[quality]\n    return no_worse and better\n\n\ndef pareto_front(table, cost, quality):\n    rows = [row for _, row in table.iterrows()]\n    return [r[\"config\"] for r in rows\n            if not any(dominates(other, r, cost, quality) for other in rows)]\n\nfront = pareto_front(efficiency, cost=\"J per question\", quality=\"accuracy\")\nprint(\"On the Pareto front:\", front)\n\nfig, ax = plt.subplots(figsize=(6, 4))\nfor _, row in efficiency.iterrows():\n    on_front = row[\"config\"] in front\n    ax.scatter(row[\"J per question\"], row[\"accuracy\"] * 100, s=80,\n               color=\"tab:green\" if on_front else \"tab:gray\")\n    ax.annotate(row[\"config\"], (row[\"J per question\"], row[\"accuracy\"] * 100),\n                textcoords=\"offset points\", xytext=(6, 4), fontsize=8)\nax.set_xlabel(\"energy per question (J)\")\nax.set_ylabel(\"accuracy (%)\")\nax.set_title(\"Green: no other configuration is both cheaper and better\")\nplt.tight_layout()\nplt.show()",
    7: "def translate(model, text, example=None, max_new_tokens=200):\n    system_content = (\"You are a professional translator. Translate the user's Galician text \"\n                      \"into fluent English.\")\n    if example is not None:\n        system_content += (\" Here is one worked example of the style expected:\\n\"\n                           f\"Galician: {example['gl']}\\nEnglish: {example['en']}\\n\"\n                           \"Output only the translation of the new text.\")\n    else:\n        system_content += \" Output only the translation.\"\n    messages = [{\"role\": \"system\", \"content\": system_content}, {\"role\": \"user\", \"content\": text}]\n    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = tokenizer(prompt, return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)\n    return tokenizer.decode(output[0][inputs[\"input_ids\"].shape[1]:], skip_special_tokens=True).strip()\n\n\nMT_CONFIGS = {\n    \"general-llm\": {\"model\": \"fp16\", \"example\": False},\n    \"general-llm-quantized\": {\"model\": \"quantized\", \"example\": False},\n    \"general-llm-retrieval-augmented\": {\"model\": \"fp16\", \"example\": True},\n}\n\nmt_runs = {}\nfor name, spec in MT_CONFIGS.items():\n    tracker = make_tracker(f\"green-3c-mt-{name}\")\n    tracker.start()\n    start_time = time.time()\n    example = retrieve_translation_example(FULL_TEXT_GL)[0] if spec[\"example\"] else None\n    translation = translate(MODELS[spec[\"model\"]], FULL_TEXT_GL, example=example)\n    elapsed = time.time() - start_time\n    emissions_kg = tracker.stop()\n    mt_runs[name] = {\"translation\": translation, \"elapsed\": elapsed,\n                     \"emissions_kg\": emissions_kg, \"energy_kwh\": energy_kwh_of(tracker)}\n    print(f\"--- {name} ({elapsed:.2f} s, {emissions_kg * 1000:.4f} g CO2eq) ---\\n{translation}\\n\")",
    8: "CANDIDATES = {name: run[\"translation\"] for name, run in mt_runs.items()}\neval_runs = {}\n\n\ndef measure_metric(metric, score_all):\n    tracker = make_tracker(f\"green-3c-eval-{metric}\")\n    tracker.start()\n    start_time = time.time()\n    try:\n        scores = score_all(CANDIDATES)\n    finally:\n        elapsed = time.time() - start_time\n        emissions_kg = tracker.stop()\n    eval_runs[metric] = {\"scores\": scores, \"elapsed\": elapsed, \"emissions_kg\": emissions_kg}\n    for name, score in scores.items():\n        print(f\"{metric:16s} {name:32s} {score}\")\n\n\nmeasure_metric(\"bleu\", lambda cands: {\n    name: round(sacrebleu.corpus_bleu([text], [[REFERENCE_EN]]).score, 2)\n    for name, text in cands.items()})\n\n\ndef bertscore_all(cands):\n    from bert_score import score as bertscore\n    _, _, f1 = bertscore(list(cands.values()), [REFERENCE_EN] * len(cands),\n                         model_type=\"distilbert-base-uncased\", verbose=False)\n    return {name: round(v, 4) for name, v in zip(cands, f1.tolist())}\n\n\ntry:\n    measure_metric(\"bertscore\", bertscore_all)\nexcept Exception as e:\n    print(f\"BERTScore failed in this environment ({type(e).__name__}: {e}). Continuing without it.\")\n\nJUDGE_INSTRUCTIONS = (\n    \"You are an expert translation evaluator. Given a source text, a candidate \"\n    \"translation, and a reference translation, rate the candidate's quality from 1 \"\n    \"(very poor) to 10 (excellent), considering both accuracy and fluency. Output \"\n    \"ONLY the number.\"\n)\n\n\ndef judge_request(candidate):\n    return (f\"Source (Galician): {FULL_TEXT_GL}\\n\\n\"\n            f\"Reference translation: {REFERENCE_EN}\\n\\n\"\n            f\"Candidate translation: {candidate}\\n\\n\"\n            \"Score (1-10):\")\n\n\ndef qwen_judge(candidate):\n    messages = [{\"role\": \"system\", \"content\": JUDGE_INSTRUCTIONS},\n                {\"role\": \"user\", \"content\": judge_request(candidate)}]\n    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = tokenizer(prompt, return_tensors=\"pt\").to(model_fp16.device)\n    with torch.no_grad():\n        output = model_fp16.generate(**inputs, max_new_tokens=10, do_sample=False)\n    return parse_score(tokenizer.decode(output[0][inputs[\"input_ids\"].shape[1]:],\n                                        skip_special_tokens=True))\n\n\nmeasure_metric(\"llm_judge_qwen\", lambda cands: {n: qwen_judge(t) for n, t in cands.items()})",
}
_fill_in = globals().get("_fill_in", set())


def _fill_in_transformer(lines):
    """An exercise cell starts with `# Exercise N`; with fill-in on for N, it becomes the solution."""
    m = re.match(r"# Exercise (\d+)\s*$", lines[0]) if lines else None
    if not m or int(m.group(1)) not in _fill_in:
        return lines
    n = int(m.group(1))
    return [f"get_ipython().set_next_input(_solutions[{n}], replace=True)\n"] + (_solutions[n] + "\n").splitlines(True)


_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup
                                     if getattr(t, "__name__", "") != "_fill_in_transformer"]
_ip.input_transformers_cleanup.append(_fill_in_transformer)


def _fill_in_callback(exercises):
    """Called by the buttons: fill-in on for these exercises."""
    _fill_in.update(exercises)
    return JSON({"ok": True})


try:
    from google.colab import output
    output.register_callback("greenai.fill_in", _fill_in_callback)
except ImportError:  # outside Colab the buttons do nothing; run _fill_in_callback([N]) instead
    pass

display(HTML("<button onclick=\"greenaiFill(this, [1, 2, 3, 4, 5, 6, 7, 8])\" data-done=\"✓ Every exercise cell you run from now on gets its solution. Runtime → Run all fills in all of them.\" data-need=\"First run this cell (▶ on its left), then click again.\" style=\"font: 500 14px Roboto, Arial, sans-serif; padding: 7px 16px; border: 0; border-radius: 4px; background: #1a73e8; color: #fff; cursor: pointer;\">Fill in solutions</button><span style=\"font: 14px Roboto, Arial, sans-serif; margin-left: 12px;\"></span>\n<script>\nasync function greenaiFill(button, exercises) {\n  const msg = button.nextElementSibling;\n  let ok = false;\n  try {\n    const r = await google.colab.kernel.invokeFunction(\"greenai.fill_in\", [exercises], {});\n    ok = Boolean(r && r.data && r.data[\"application/json\"] && r.data[\"application/json\"].ok);\n  } catch (e) {}\n  msg.textContent = ok ? button.dataset.done : button.dataset.need;\n}\n</script>"))

## Step 0 — Install and import

Colab already includes `torch`, `transformers`, `accelerate`, `sentence-transformers`,
`scikit-learn`, `pandas` and `matplotlib`; the cell installs only what is missing.
`gptqmodel` (with `optimum`) runs the GPTQ path, as in Notebook 2, and `bitsandbytes` the
fallback path. Building `gptqmodel` can take a few minutes.

In [ ]:
!pip install -q bitsandbytes optimum codecarbon sacrebleu bert-score

import importlib.util
if importlib.util.find_spec("gptqmodel") is None:
    !pip install -q gptqmodel && echo "gptqmodel installed." || echo "gptqmodel failed to install. The quantized model will fall back to bitsandbytes int8."
else:
    print("gptqmodel already installed, skipping.")

**After the install cell finishes, restart once:** `Runtime` → `Restart session`. This keeps
the installed packages and only clears the running Python process. Installing `gptqmodel`
can bring in a different `numpy` build than the one already loaded, which otherwise shows
up a few cells down as a confusing `numpy` or `ModuleNotFoundError` on import. After the
restart, continue from the next cell.

In [ ]:
import os
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sacrebleu
import torch
from codecarbon import EmissionsTracker
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GPTQConfig

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. This notebook needs a GPU runtime for the quantized model: "
          "Runtime > Change runtime type > T4 GPU.")


def make_tracker(project_name):
    return EmissionsTracker(project_name=project_name, measure_power_secs=1,
                            save_to_file=False, log_level="warning")


def energy_kwh_of(tracker):
    try:
        return float(tracker.final_emissions_data.energy_consumed)
    except (AttributeError, TypeError):
        return None


def gpu_memory_gb():
    return torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else None


def parse_score(text):
    match = re.search(r"\d+(\.\d+)?", str(text))
    return float(match.group()) if match else None

## Step 1 — The corpus and the quiz

Notebook 3's 26 documents and 26 questions, unchanged: Notebook 1's eight questions first,
then one per new document. In this notebook the documents are used only by the retriever.

In [ ]:
CORPUS = {
    "training_vs_inference": """
Training a large language model is a one-time, very expensive event: the organization
that builds the model spends a large amount of compute once. Inference — actually using
the trained model to answer a query — is comparatively cheap per request, but it happens
every single time anyone uses the model. Once a model is deployed and used by many people,
the accumulated cost of inference tends to dominate the model's total lifetime footprint,
even though any single inference looks negligible next to the training run.
Source: Patterson et al., "Carbon Emissions and Large Neural Network Training," 2021
(arXiv:2104.10350). Training GPT-3 was estimated to emit 552 metric tons of CO2-equivalent,
using about 1,287 megawatt-hours of electricity.
""",
    "data_centers": """
Data centers are the physical facilities that house the servers running most of the
world's AI models. According to the International Energy Agency, global data center
electricity consumption was about 415 terawatt-hours in 2024, roughly 1.5 percent of all
electricity consumed worldwide that year. That demand is growing fast: electricity use by
data centers rose by about 17 percent in 2025 alone, driven mostly by AI workloads, and is
projected to approach 950 terawatt-hours by 2030, or about 3 percent of global electricity
demand.
Source: International Energy Agency, "Energy and AI" and related data centre electricity
reports, 2024-2025.
""",
    "quantization_basics": """
Neural network weights are normally stored as 32-bit floating point numbers (FP32).
Quantization reduces the number of bits used to represent each weight — for example to
16-bit (FP16/BF16), 8-bit integers (INT8), or even lower. Fewer bits per number means a
smaller model file and less memory traffic when running it, at the cost of some numerical
precision. A useful analogy: quantizing a model is like reducing the number of colors used
to represent an image — the picture is still recognizable, but some fine detail is lost.
Whether this actually saves energy in practice depends on the hardware: it helps most when
the bottleneck is moving data in and out of memory, not raw computation.
Source: Grootendorst, "A Visual Guide to Quantization," 2024.
""",
    "measuring_carbon": """
Estimating the carbon footprint of running a model requires several assumptions: how much
electricity the hardware draws, the Power Usage Effectiveness (PUE) of the data center
(extra power spent on cooling and overhead), and the carbon intensity of the local
electricity grid — how much CO2 is emitted per unit of electricity, which varies a lot by
region and by time of day. Because of all these assumptions, published carbon estimates can
vary widely, and some influential early estimates have later been revised — sometimes by an
order of magnitude — once other researchers re-examined the methodology. This is one reason
this course emphasizes measuring your own numbers.
""",
    "inference_cost_task_type": """
Models doing the same task can differ widely in what they cost to run. A 2024 study
compared task-specific models (built and trained for one narrow job) against multi-purpose
generative models (like general-purpose chatbots) performing the exact same tasks. For
extractive question answering, the most efficient task-specific models emitted around 0.3
grams of CO2-equivalent per 1,000 inferences, while general-purpose generative models
emitted around 10 grams of CO2-equivalent per 1,000 inferences for the same task — roughly
33 times more. The gap was consistent across several other tasks too, even when
controlling for the number of parameters.
Source: Luccioni, Jernite & Strubell, "Power Hungry Processing: Watts Driving the Cost of AI
Deployment?", FAccT 2024 (arXiv:2311.16863).
""",
    "rag_concept": """
Retrieval-Augmented Generation, or RAG, is a technique where a model is given relevant
external text — retrieved from a document collection — as part of its input at query time,
instead of relying only on what it "learned" during training. This lets a smaller, cheaper
model answer questions about information it was never trained on, without the cost of
retraining or fine-tuning a much larger model on that information. The retrieval step
usually uses a separate, lightweight embedding model to find which documents are relevant
before the language model ever sees them.
""",
    "small_models_trend": """
Between 2024 and 2026, there has been a strong trend toward small, efficient open-weight
language models (often called SLMs, small language models) in the 1-4 billion parameter
range, designed to run on a single consumer GPU or even a laptop CPU after quantization.
These models are trained to be very capable for their size, and combined with quantization
and retrieval, they let many applications avoid depending on a large, general-purpose
model at all — often at a fraction of the inference cost, if the task doesn't need the
breadth of a much bigger model.
""",
    "transformer_basics": """
Modern language models are built from transformer layers. Inside each layer,
"self-attention" lets every position in the input sequence gather information from every
other position, which is what allows the model to relate words like "it" back to the noun
it refers to several words earlier. This mechanism is powerful but has a computational cost
that grows faster than the length of the input (roughly quadratically with sequence
length), and more parameters mean more numbers to multiply and move through memory for
every single token generated. This is the underlying reason why both model size and input
length are direct levers on energy cost.
""",
    "nas_architecture_search": """
A widely repeated 2019 claim held that training a particular large NLP model produced
roughly as much lifetime CO2-equivalent as five cars. The model came from a neural
architecture search: instead of a human hand-designing a network's structure, an
algorithm trains and evaluates thousands of candidate architectures before selecting a
winner, multiplying the effective training cost many times over. The search itself was
hunting for a better machine translation architecture. The original "five cars" estimate
was later shown to be roughly 10 times too high, once other researchers re-examined the
hardware and power assumptions behind it.
Source: Strubell et al., "Energy and Policy Considerations for Deep Learning in NLP," ACL
2019; So, Le & Liang, "The Evolved Transformer," 2019.
""",
    "nlp_market_size": """
Market-research estimates of the global NLP industry's size vary widely even for the same
year. One widely-cited 2025 estimate puts the global NLP market at roughly $37 billion,
projected to reach about $193 billion by 2034 — about 20 percent annual growth. Other
firms estimate the same year's market anywhere from about $25 billion to well above $60
billion. The spread exists because "NLP market" is a loosely defined category: analysts
include or exclude voice assistants, chatbot platforms, search, and broader "AI" spending
differently, and each firm's method is rarely published in full.
Source: Fortune Business Insights, "Natural Language Processing Market," 2025.
""",
    "mt_history": """
Machine translation is one of AI's oldest stated goals. The first public demonstration was
the Georgetown-IBM experiment in 1954, translating a small set of Russian sentences into
English, received with considerable, later unrealistic, optimism about how soon
translation would be "solved." In 1966 the U.S. government's ALPAC report concluded
machine translation was falling short of its promises and recommended cutting funding,
setting back American MT research roughly a decade. Progress resumed with rule-based
systems, then statistical systems that learned patterns directly from translated text;
Google Translate launched on statistical MT in 2006.
Source: Hutchins, "Machine Translation: A Concise History," 2007.
""",
    "attention_origin": """
Early neural machine translation systems compressed an entire source sentence into a
single fixed-size vector before generating a translation from that one vector alone — a
bottleneck that hurt quality as sentences got longer, since detail from earlier in the
sentence was easily lost. Bahdanau, Cho and Bengio fixed this in 2015 by letting the
decoder look back at every word of the source sentence at each generation step and learn
which source words matter most for the word currently being produced. They called this
mechanism attention. It was invented specifically to solve a machine translation problem,
years before it became the core building block of modern language models.
Source: Bahdanau, Cho & Bengio, "Neural Machine Translation by Jointly Learning to Align
and Translate," ICLR 2015.
""",
    "transformer_self_attention": """
In 2017, Vaswani and colleagues showed that the attention mechanism originally built for
translating between two languages also works within a single sentence: every word can
attend to every other word in the same sentence, including itself, a pattern called
self-attention. This lets a model resolve relationships like a pronoun referring back to a
noun several words earlier, without any hand-written grammatical rule. Their architecture,
the transformer, built entirely around this mechanism plus a small feed-forward network
per layer, is the architecture behind essentially every modern large language model.
Source: Vaswani et al., "Attention Is All You Need," NeurIPS 2017.
""",
    "skipgram_embeddings": """
Word embeddings — dense numerical vectors that place semantically similar words close
together — are learned from text. The skip-gram model, introduced by Mikolov and
colleagues, trains a network on a simple task: given one word, predict the words that tend
to appear near it in real text. The network's internal weight matrix, learned purely to
get better at that prediction task, becomes the word embeddings as a side effect. Nobody
tells the model what "similar" means; words that occur in similar contexts end up needing
similar internal representations to succeed at the same prediction task.
Source: Mikolov et al., "Efficient Estimation of Word Representations in Vector Space,"
2013.
""",
    "gpt_scaling": """
Until the late 2010s, NLP typically meant a separate, purpose-built model for each task:
one system for translation, another for summarization, another for sentiment. Radford and
colleagues showed in 2019, with GPT-2, that a single model trained only to predict the
next word of text could be redirected to many different tasks just by phrasing the request
differently — "prompting" it — without retraining it. Brown and colleagues confirmed
this worked at much larger scale in 2020 with GPT-3. This is broadly considered the moment
public conversation stopped clearly distinguishing "NLP" from "AI."
Source: Radford et al., "Language Models are Unsupervised Multitask Learners," 2019; Brown
et al., "Language Models are Few-Shot Learners," NeurIPS 2020.
""",
    "galician_llm_gap": """
Galician's presence in the training data of mainstream large language models has been
described in the research literature as "almost residual" — a very small fraction,
sitting close enough to Spanish and Portuguese that general-purpose models tend to blur
the three together instead of reliably producing correct, distinctly Galician text. One
response has been to build smaller, dedicated models: a 1.3-billion-parameter model
trained specifically on Galician and Portuguese text outperformed a much larger
general-purpose base model specifically at producing correct Galician.
Source: Gamallo et al., "A Galician-Portuguese Generative Model," EPIA 2024; Proxecto Nós,
Universidade de Santiago de Compostela.
""",
    "green_mt_knight": """
Some Green NLP research targets machine translation directly. One recent MT-specific
efficiency recipe, nicknamed "The Green KNIGHT," reached up to 140 times faster inference
and about 98.7 percent less energy than a baseline system, for roughly a 10 to 12 percent
relative cost in translation quality. Efficiency techniques are frequently developed and
tested in machine translation before being generalized to language models more broadly.
Source: Guta, Petrick & Polák, "The Green KNIGHT," EMNLP Findings 2025.
""",
    "mt_quantization": """
Quantization — reducing the number of bits used to store and compute with a model's
numbers — has been studied specifically in the context of machine translation systems.
Testing an efficiency technique on the specific task it will serve is consistent with this
course's broader theme: measure the technique on the task you actually care about.
Source: Shterionov & Vanmassenhove, 2022.
""",
    "quantization_energy_quality": """
A 2025 study systematically measured quantization's effects across several large language
models on production-grade GPUs. Quantization generally saved energy — up to about 30
percent versus full precision — and in some tests cut latency by as much as 70 percent.
But pushed too far, quality did not degrade gently: on a code-generation benchmark, one
13-billion-parameter model lost up to 92 percent of its pass rate under aggressive
quantization. The same study found that smaller models are more vulnerable to this kind of
quality collapse than larger ones.
Source: Shi & Ding, "Systematic Characterization of LLM Quantization: A Performance,
Energy, and Quality Perspective," 2025 (arXiv:2508.16712).
""",
    "quantization_hardware_dependency": """
A separate 2025 study tested 28 quantized language models, including the Qwen2.5 family,
on edge hardware. Its finding: 8-bit and 4-bit quantization roughly halved energy use
with only a modest accuracy cost, but only when the underlying hardware actually had
efficient support for low-precision arithmetic. Pushing further, to 3-bit quantization,
sometimes increased energy use instead of decreasing it, because the hardware spent more
effort unpacking the unusual bit-width than it saved. The benefit of quantization depends
on the chip it runs on.
Source: Husom et al., "Sustainable LLM Inference for Edge AI," 2025 (arXiv:2504.03360).
""",
    "gpu_precision_throughput": """
On many GPUs, lower numerical precision also runs faster, on top of shrinking file size
and memory use, because the hardware has separate circuits tuned for different
bit-widths. The NVIDIA T4 GPU, commonly available on free-tier cloud notebooks, has a peak
throughput of about 8.1 TFLOPS at 32-bit floating point, 65 TFLOPS at 16-bit, 130 TOPS at
8-bit integer, and 260 TOPS at 4-bit integer — roughly doubling with each step down in
precision, on the exact same physical chip.
Source: NVIDIA T4 Tensor Core GPU datasheet.
""",
    "rag_paper": """
The technique now generally called retrieval-augmented generation was formalized in a 2020
paper: instead of relying only on facts baked into a model's parameters during training,
retrieve relevant external text from a document collection at query time and condition the
model's generated answer on that retrieved text. This lets a comparatively small, cheap
model answer questions about information it was never trained on, without the cost of
retraining or fine-tuning a much larger model to memorize that information.
Source: Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks,"
NeurIPS 2020 (arXiv:2005.11401).
""",
    "rag_vs_long_context": """
A 2024 study directly compared retrieval-augmented generation against long-context
prompting — feeding the whole text into a model with a large enough context window —
using three strong language models: GPT-4o, GPT-3.5-Turbo and Gemini-1.5-Pro. For all
three, long-context prompting gave better answers on average. But retrieval read far less
text: about 1,500 tokens of retrieved passages per question, against 10,000 to 100,000
tokens for the full texts. For 63 percent of the queries, the two approaches produced
identical answers. A combined method, Self-Route, tries retrieval first and switches to
long context only when the model judges the retrieved text insufficient; it cut cost by
65 percent for Gemini-1.5-Pro and 39 percent for GPT-4o while staying close to
long-context quality.
Source: Li, Li, Zhang, Mei & Bendersky, "Retrieval Augmented Generation or Long-Context
LLMs? A Comprehensive Study and Hybrid Approach," EMNLP 2024 Industry Track
(arXiv:2407.16833).
""",
    "knn_mt": """
Retrieval has also been applied directly to machine translation. One method augments an
already-trained translation model with a large datastore of previously translated example
sentence pairs. At each step of generating a translation, the system looks up the nearest
neighbors in that datastore and blends their suggested next word in with the trained
model's own prediction. This lets a translation system incorporate new examples without
retraining, in the same spirit as retrieval-augmented generation for question answering,
applied to translation instead.
Source: Khandelwal, Fan, Jurafsky, Zettlemoyer & Lewis, "Nearest Neighbor Machine
Translation," ICLR 2021.
""",
    "water_footprint": """
Electricity is only one of the resources AI systems consume. Data centers also use water,
mainly for cooling. Google has estimated that a single prompt to its Gemini model consumes
about 0.26 milliliters of water — roughly five drops. That figure is small per prompt but,
like electricity, scales with the number of queries a system serves, and water use is
generally reported and discussed far less often than energy or carbon figures are.
Source: Google, water-footprint estimate for Gemini prompts.
""",
    "image_generation_cost": """
Which task a model performs is itself one of the biggest levers on energy cost,
independent of any efficiency technique applied to the model. A 2024 study found that
image generation averaged about 2.9 kilowatt-hours per 1,000 inferences — roughly 1,450
times the energy of the cheapest text-based task measured in the same study. Choosing a
smaller, cheaper task, or a task-specific tool instead of a general-purpose one, can matter
as much as any single optimization applied afterward.
Source: Luccioni, Jernite & Strubell, "Power Hungry Processing: Watts Driving the Cost of
AI Deployment?", FAccT 2024 (arXiv:2311.16863).
""",
}

for name, text in CORPUS.items():
    print(f"{name:34s} {len(text.split()):4d} words")
print(f"\nTotal documents: {len(CORPUS)}, "
      f"total words: {sum(len(t.split()) for t in CORPUS.values())}")

In [ ]:
QA_PAIRS = [
    {
        "question": "According to Patterson et al. (2021), how many metric tons of CO2-equivalent did training GPT-3 emit?",
        "reference": ["552"],
        "source": "training_vs_inference",
    },
    {
        "question": "Per the IEA, roughly what percentage of global electricity consumption did data centers represent in 2024?",
        "reference": ["1.5"],
        "source": "data_centers",
    },
    {
        "question": "What does the acronym PUE stand for?",
        "reference": ["power usage effectiveness"],
        "source": "measuring_carbon",
    },
    {
        "question": "What does carbon footprint estimation depend on besides hardware power draw and grid carbon intensity?",
        "reference": ["pue", "power usage effectiveness"],
        "source": "measuring_carbon",
    },
    {
        "question": "In the 2024 study on inference costs, about how many grams of CO2-equivalent do the most efficient task-specific models emit per 1,000 extractive question-answering inferences?",
        "reference": ["0.3"],
        "source": "inference_cost_task_type",
    },
    {
        "question": "What is the name of the technique where a model is given retrieved external text at query time instead of relying only on what it learned during training?",
        "reference": ["retrieval-augmented generation", "rag"],
        "source": "rag_concept",
    },
    {
        "question": "What size range (in billions of parameters) do small language models (SLMs) typically fall into?",
        "reference": ["1-4", "1 to 4", "1-4 billion"],
        "source": "small_models_trend",
    },
    {
        "question": "What mechanism inside a transformer layer lets every position gather information from every other position?",
        "reference": ["self-attention", "attention"],
        "source": "transformer_basics",
    },
    {
        "question": "The original '5 cars' carbon estimate for the architecture-search model was later shown to be roughly how many times too high?",
        "reference": ["10", "ten"],
        "source": "nas_architecture_search",
    },
    {
        "question": "Per Fortune Business Insights, roughly what is the projected global NLP market size by 2034?",
        "reference": ["193"],
        "source": "nlp_market_size",
    },
    {
        "question": "In what year did the Georgetown-IBM machine translation demonstration take place?",
        "reference": ["1954"],
        "source": "mt_history",
    },
    {
        "question": "In what year did Bahdanau, Cho and Bengio introduce the attention mechanism to fix early neural MT's single-vector bottleneck?",
        "reference": ["2015"],
        "source": "attention_origin",
    },
    {
        "question": "What is the name of the architecture introduced by Vaswani et al. in 2017, built entirely around self-attention?",
        "reference": ["transformer"],
        "source": "transformer_self_attention",
    },
    {
        "question": "What is the name of the model introduced by Mikolov et al. that produces word embeddings as a byproduct of predicting nearby words?",
        "reference": ["skip-gram", "skipgram"],
        "source": "skipgram_embeddings",
    },
    {
        "question": "In what year did Brown et al. show GPT-3 could perform many tasks via prompting at large scale?",
        "reference": ["2020"],
        "source": "gpt_scaling",
    },
    {
        "question": "How many billion parameters does the dedicated Galician-Portuguese model described in Gamallo et al. have?",
        "reference": ["1.3"],
        "source": "galician_llm_gap",
    },
    {
        "question": "According to the Green KNIGHT study, roughly what percentage energy reduction did their MT-specific recipe achieve?",
        "reference": ["98.7"],
        "source": "green_mt_knight",
    },
    {
        "question": "What NLP task was quantization tested on specifically, per Shterionov and Vanmassenhove (2022)?",
        "reference": ["machine translation", "mt"],
        "source": "mt_quantization",
    },
    {
        "question": "Per Shi and Ding (2025), up to what percentage drop in code-generation pass rate did aggressive quantization cause for a 13B model?",
        "reference": ["92"],
        "source": "quantization_energy_quality",
    },
    {
        "question": "Per Husom et al. (2025), which specific model family was tested for quantization energy efficiency on edge hardware?",
        "reference": ["qwen2.5", "qwen"],
        "source": "quantization_hardware_dependency",
    },
    {
        "question": "According to the NVIDIA T4 datasheet, what is the GPU's peak INT8 throughput in TOPS?",
        "reference": ["130"],
        "source": "gpu_precision_throughput",
    },
    {
        "question": "In what year did Lewis et al. introduce the original Retrieval-Augmented Generation (RAG) framework?",
        "reference": ["2020"],
        "source": "rag_paper",
    },
    {
        "question": "In the Li et al. (2024) study, for what percentage of queries did RAG and long-context prompting produce identical answers?",
        "reference": ["63"],
        "source": "rag_vs_long_context",
    },
    {
        "question": "What is the common short name for the retrieval-augmented machine translation method introduced by Khandelwal et al. in 2021?",
        "reference": ["knn-mt", "knn"],
        "source": "knn_mt",
    },
    {
        "question": "Roughly how many milliliters of water does Google estimate a single Gemini prompt consumes?",
        "reference": ["0.26"],
        "source": "water_footprint",
    },
    {
        "question": "Per Luccioni et al. (2024), roughly how many kilowatt-hours per 1,000 inferences does image generation average?",
        "reference": ["2.9"],
        "source": "image_generation_cost",
    },
]

N_SESSION1_QUESTIONS = 8

assert all(pair["source"] in CORPUS for pair in QA_PAIRS)
print(f"Number of questions: {len(QA_PAIRS)}")

## Step 2 — Load both models

Both models stay in GPU memory for the whole notebook, so every run below uses the same
loaded weights. The fp16 model is the baseline and also serves retrieval. The quantized
model is Notebook 2's: the official GPTQ int4 checkpoint, with bitsandbytes int8 as the
fallback if GPTQ fails to load. Memory is measured before and after each load, since a
smaller memory footprint is the one benefit of quantization that shows up on any GPU.

### ✏️ Exercise 1 — How much GPU memory do the weights take?

In `load_timed`, record `weights_gb`: the GPU memory in use after loading minus the
memory in use before, rounded to 2 decimals. On a CPU runtime both readings are `None`;
then `weights_gb` is `None` too.

<details><summary><b>Hints</b> (click to open)</summary>

- `gpu_memory_gb()` was read once before loading (`mem_before`) and once after (`mem_after`).

</details>

In [ ]:
from IPython.display import HTML, display

display(HTML("<button onclick=\"greenaiFill(this, [1])\" data-done=\"✓ Now run the exercise cell below: the solution replaces its code.\" data-need=\"First run the cell &quot;Run this cell first&quot; under the instructions, then click again.\" style=\"font: 500 14px Roboto, Arial, sans-serif; padding: 7px 16px; border: 0; border-radius: 4px; background: #1a73e8; color: #fff; cursor: pointer;\">Fill in solution</button><span style=\"font: 14px Roboto, Arial, sans-serif; margin-left: 12px;\"></span>\n<script>\nasync function greenaiFill(button, exercises) {\n  const msg = button.nextElementSibling;\n  let ok = false;\n  try {\n    const r = await google.colab.kernel.invokeFunction(\"greenai.fill_in\", [exercises], {});\n    ok = Boolean(r && r.data && r.data[\"application/json\"] && r.data[\"application/json\"].ok);\n  } catch (e) {}\n  msg.textContent = ok ? button.dataset.done : button.dataset.need;\n}\n</script>"))

In [ ]:
# Exercise 1
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
GPTQ_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
load_info = {}


def load_timed(label, loader):
    mem_before = gpu_memory_gb()
    start = time.time()
    loaded = loader()
    loaded.eval()
    loaded.generation_config.do_sample = False
    loaded.generation_config.temperature = None
    loaded.generation_config.top_p = None
    loaded.generation_config.top_k = None
    mem_after = gpu_memory_gb()
    load_info[label] = {
        "load_seconds": round(time.time() - start, 1),
        "weights_gb": ___,  # TODO: memory added by loading, rounded to 2 decimals; None on CPU
    }
    return loaded


model_fp16 = load_timed("fp16", lambda: AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, dtype=torch.float16, device_map="auto"))

try:
    model_quant = load_timed("quantized", lambda: AutoModelForCausalLM.from_pretrained(
        GPTQ_MODEL_NAME, device_map="auto", quantization_config=GPTQConfig(bits=4, backend="torch")))
    QUANT_CONFIG, QUANT_MODEL_NAME = "gptq-int4", GPTQ_MODEL_NAME
except Exception as e:
    print(f"GPTQ load failed ({type(e).__name__}: {e}). Falling back to bitsandbytes int8.")
    model_quant = load_timed("quantized", lambda: AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME, quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        dtype=torch.float16, device_map="auto"))
    QUANT_CONFIG, QUANT_MODEL_NAME = "bnb-int8", BASE_MODEL_NAME

MODELS = {"fp16": model_fp16, "quantized": model_quant}
print(f"Quantized path: {QUANT_CONFIG}")
pd.DataFrame(load_info).T

In [ ]:
# Check for Exercise 1. Run this cell as is.
_w = {k: v["weights_gb"] for k, v in load_info.items()}
assert set(_w) == {"fp16", "quantized"}
assert _w["fp16"] is None or 0 < _w["quantized"] < _w["fp16"], f"the quantized weights should be smaller: {_w}"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
GPTQ_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
load_info = {}


def load_timed(label, loader):
    mem_before = gpu_memory_gb()
    start = time.time()
    loaded = loader()
    loaded.eval()
    loaded.generation_config.do_sample = False
    loaded.generation_config.temperature = None
    loaded.generation_config.top_p = None
    loaded.generation_config.top_k = None
    mem_after = gpu_memory_gb()
    load_info[label] = {
        "load_seconds": round(time.time() - start, 1),
        "weights_gb": None if mem_after is None else round(mem_after - mem_before, 2),
    }
    return loaded


model_fp16 = load_timed("fp16", lambda: AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, dtype=torch.float16, device_map="auto"))

try:
    model_quant = load_timed("quantized", lambda: AutoModelForCausalLM.from_pretrained(
        GPTQ_MODEL_NAME, device_map="auto", quantization_config=GPTQConfig(bits=4, backend="torch")))
    QUANT_CONFIG, QUANT_MODEL_NAME = "gptq-int4", GPTQ_MODEL_NAME
except Exception as e:
    print(f"GPTQ load failed ({type(e).__name__}: {e}). Falling back to bitsandbytes int8.")
    model_quant = load_timed("quantized", lambda: AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME, quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        dtype=torch.float16, device_map="auto"))
    QUANT_CONFIG, QUANT_MODEL_NAME = "bnb-int8", BASE_MODEL_NAME

MODELS = {"fp16": model_fp16, "quantized": model_quant}
print(f"Quantized path: {QUANT_CONFIG}")
pd.DataFrame(load_info).T
```

</details>

## Step 3 — The retriever

Notebook 3's dense retriever: the multilingual MiniLM sentence encoder, 26 documents
embedded once, and cosine similarity to pick the `TOP_K = 2` closest documents for each
question. The same encoder retrieves translation-memory examples in Part 2.

In [ ]:
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

DOC_NAMES = list(CORPUS)
DOC_TEXTS = [CORPUS[name].strip() for name in DOC_NAMES]

tracker_index = make_tracker("green-3c-index")
tracker_index.start()
doc_embeddings = embedder.encode(DOC_TEXTS, normalize_embeddings=True)
emissions_index_kg = tracker_index.stop()

TOP_K = 2


def retrieve(question, k=TOP_K):
    q_emb = embedder.encode([question], normalize_embeddings=True)[0]
    sims = doc_embeddings @ q_emb
    return [DOC_NAMES[i] for i in np.argsort(-sims)[:k]]


print(f"Indexed {len(DOC_NAMES)} documents ({emissions_index_kg * 1000:.6f} g CO2eq, once).")

## Step 4 — One pipeline, three configurations

`answer_question` takes the model and the list of documents to include. An empty list
gives the closed-book prompt: the question alone, with the same one-sentence instruction.

### ✏️ Exercise 2 — One pipeline, three configurations

Two gaps:

1. In `answer_question`, `n_prompt` is the number of prompt tokens, used to cut the prompt
   off the model's output.
2. Fill in `CONFIGS`. Each configuration is a dict with `"model"` (`"fp16"` or
   `"quantized"`, the keys of `MODELS`) and `"retrieval"` (`True` or `False`).

<details><summary><b>Hints</b> (click to open)</summary>

- `inputs["input_ids"]` has shape (batch size, prompt length).
- Closed book means no documents: `"retrieval": False`.

</details>

In [ ]:
from IPython.display import HTML, display

display(HTML("<button onclick=\"greenaiFill(this, [2])\" data-done=\"✓ Now run the exercise cell below: the solution replaces its code.\" data-need=\"First run the cell &quot;Run this cell first&quot; under the instructions, then click again.\" style=\"font: 500 14px Roboto, Arial, sans-serif; padding: 7px 16px; border: 0; border-radius: 4px; background: #1a73e8; color: #fff; cursor: pointer;\">Fill in solution</button><span style=\"font: 14px Roboto, Arial, sans-serif; margin-left: 12px;\"></span>\n<script>\nasync function greenaiFill(button, exercises) {\n  const msg = button.nextElementSibling;\n  let ok = false;\n  try {\n    const r = await google.colab.kernel.invokeFunction(\"greenai.fill_in\", [exercises], {});\n    ok = Boolean(r && r.data && r.data[\"application/json\"] && r.data[\"application/json\"].ok);\n  } catch (e) {}\n  msg.textContent = ok ? button.dataset.done : button.dataset.need;\n}\n</script>"))

In [ ]:
# Exercise 2
def build_prompt(question, doc_names):
    if not doc_names:
        messages = [
            {"role": "system",
             "content": "You are a concise assistant. Answer the question in one short sentence."},
            {"role": "user", "content": f"Question: {question}"},
        ]
    else:
        context = "\n\n".join(f"[{name}]\n{CORPUS[name].strip()}" for name in doc_names)
        messages = [
            {"role": "system",
             "content": ("You are a concise assistant. Answer the question using ONLY the "
                         "documents provided below. Answer in one short sentence.")},
            {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {question}"},
        ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def answer_question(model, question, doc_names, max_new_tokens=60):
    inputs = tokenizer(build_prompt(question, doc_names), return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    n_prompt = ___  # TODO: number of tokens in the prompt
    return tokenizer.decode(output[0][n_prompt:], skip_special_tokens=True).strip(), n_prompt


def score_answer(answer, reference_list):
    return any(ref.lower() in answer.lower() for ref in reference_list)


CONFIGS = {
    "baseline-fp16-closed-book": ___,  # TODO: fp16 model, no documents
    f"{QUANT_CONFIG}-closed-book": ___,  # TODO: quantized model, no documents
    f"retrieval-top{TOP_K}-fp16": ___,  # TODO: fp16 model, retrieved documents
}

q = QA_PAIRS[0]["question"]
print("Q:", q)
for config, spec in CONFIGS.items():
    docs = retrieve(q) if spec["retrieval"] else []
    print(f"  {config:28s} {answer_question(MODELS[spec['model']], q, docs)[0]}")

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert CONFIGS["baseline-fp16-closed-book"] == {"model": "fp16", "retrieval": False}
assert CONFIGS[f"{QUANT_CONFIG}-closed-book"] == {"model": "quantized", "retrieval": False}
assert CONFIGS[f"retrieval-top{TOP_K}-fp16"] == {"model": "fp16", "retrieval": True}
_a, _n = answer_question(model_fp16, "What is 2 + 2?", [], max_new_tokens=8)
assert "Question:" not in _a and 10 < _n < 80, (_a, _n)
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
def build_prompt(question, doc_names):
    if not doc_names:
        messages = [
            {"role": "system",
             "content": "You are a concise assistant. Answer the question in one short sentence."},
            {"role": "user", "content": f"Question: {question}"},
        ]
    else:
        context = "\n\n".join(f"[{name}]\n{CORPUS[name].strip()}" for name in doc_names)
        messages = [
            {"role": "system",
             "content": ("You are a concise assistant. Answer the question using ONLY the "
                         "documents provided below. Answer in one short sentence.")},
            {"role": "user", "content": f"Documents:\n{context}\n\nQuestion: {question}"},
        ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def answer_question(model, question, doc_names, max_new_tokens=60):
    inputs = tokenizer(build_prompt(question, doc_names), return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    n_prompt = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0][n_prompt:], skip_special_tokens=True).strip(), n_prompt


def score_answer(answer, reference_list):
    return any(ref.lower() in answer.lower() for ref in reference_list)


CONFIGS = {
    "baseline-fp16-closed-book": {"model": "fp16", "retrieval": False},
    f"{QUANT_CONFIG}-closed-book": {"model": "quantized", "retrieval": False},
    f"retrieval-top{TOP_K}-fp16": {"model": "fp16", "retrieval": True},
}

q = QA_PAIRS[0]["question"]
print("Q:", q)
for config, spec in CONFIGS.items():
    docs = retrieve(q) if spec["retrieval"] else []
    print(f"  {config:28s} {answer_question(MODELS[spec['model']], q, docs)[0]}")
```

</details>

## Step 5 — Measure: the full quiz, three times

Each configuration answers all 26 questions under its own CodeCarbon tracker. For the
retrieval run, retrieving the documents happens inside the tracker, so its cost counts.

### ✏️ Exercise 3 — Run the quiz for each configuration

Inside the loop: fetch documents only when the configuration uses retrieval, and
score each answer against the question's reference keywords.

<details><summary><b>Hints</b> (click to open)</summary>

- `spec["retrieval"]` is True or False for the current configuration.
- `score_answer(answer, reference_list)` is defined two cells above.

</details>

In [ ]:
from IPython.display import HTML, display

display(HTML("<button onclick=\"greenaiFill(this, [3])\" data-done=\"✓ Now run the exercise cell below: the solution replaces its code.\" data-need=\"First run the cell &quot;Run this cell first&quot; under the instructions, then click again.\" style=\"font: 500 14px Roboto, Arial, sans-serif; padding: 7px 16px; border: 0; border-radius: 4px; background: #1a73e8; color: #fff; cursor: pointer;\">Fill in solution</button><span style=\"font: 14px Roboto, Arial, sans-serif; margin-left: 12px;\"></span>\n<script>\nasync function greenaiFill(button, exercises) {\n  const msg = button.nextElementSibling;\n  let ok = false;\n  try {\n    const r = await google.colab.kernel.invokeFunction(\"greenai.fill_in\", [exercises], {});\n    ok = Boolean(r && r.data && r.data[\"application/json\"] && r.data[\"application/json\"].ok);\n  } catch (e) {}\n  msg.textContent = ok ? button.dataset.done : button.dataset.need;\n}\n</script>"))

In [ ]:
# Exercise 3
runs = {}
for config, spec in CONFIGS.items():
    model = MODELS[spec["model"]]
    tracker = make_tracker(f"green-3c-{config}")
    rows = []
    tracker.start()
    start_time = time.time()
    for pair in QA_PAIRS:
        docs = ___  # TODO: retrieved document names, or [] for closed book
        answer, n_prompt = answer_question(model, pair["question"], docs)
        rows.append({**pair, "docs": docs, "answer": answer, "prompt_tokens": n_prompt,
                     "correct": ___,  # TODO: is the answer correct?
                     "retrieval_correct": pair["source"] in docs if spec["retrieval"] else None})
    elapsed = time.time() - start_time
    emissions_kg = tracker.stop()
    runs[config] = {"rows": rows, "elapsed": elapsed, "emissions_kg": emissions_kg,
                    "energy_kwh": energy_kwh_of(tracker), **spec}
    print(f"{config:28s} {elapsed:6.1f} s  {emissions_kg * 1000:.4f} g CO2eq")

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert all(r["docs"] == [] for r in runs["baseline-fp16-closed-book"]["rows"])
assert all(len(r["docs"]) == TOP_K for r in runs[f"retrieval-top{TOP_K}-fp16"]["rows"])
_r = runs[f"retrieval-top{TOP_K}-fp16"]["rows"][0]
assert _r["correct"] == score_answer(_r["answer"], _r["reference"])
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
runs = {}
for config, spec in CONFIGS.items():
    model = MODELS[spec["model"]]
    tracker = make_tracker(f"green-3c-{config}")
    rows = []
    tracker.start()
    start_time = time.time()
    for pair in QA_PAIRS:
        docs = retrieve(pair["question"]) if spec["retrieval"] else []
        answer, n_prompt = answer_question(model, pair["question"], docs)
        rows.append({**pair, "docs": docs, "answer": answer, "prompt_tokens": n_prompt,
                     "correct": score_answer(answer, pair["reference"]),
                     "retrieval_correct": pair["source"] in docs if spec["retrieval"] else None})
    elapsed = time.time() - start_time
    emissions_kg = tracker.stop()
    runs[config] = {"rows": rows, "elapsed": elapsed, "emissions_kg": emissions_kg,
                    "energy_kwh": energy_kwh_of(tracker), **spec}
    print(f"{config:28s} {elapsed:6.1f} s  {emissions_kg * 1000:.4f} g CO2eq")
```

</details>

## Step 6 — Score and compare

Answers side by side, then one summary table. Accuracy on Notebook 1's eight questions is
shown separately, since those are the questions every earlier session used.

### ✏️ Exercise 4 — Summary per configuration

Fill in two rows of the summary: answer accuracy (fraction of correct answers) and
emissions per question, in grams.

<details><summary><b>Hints</b> (click to open)</summary>

- 1 kg = 1000 g.
- Divide by the number of questions.

</details>

In [ ]:
from IPython.display import HTML, display

display(HTML("<button onclick=\"greenaiFill(this, [4])\" data-done=\"✓ Now run the exercise cell below: the solution replaces its code.\" data-need=\"First run the cell &quot;Run this cell first&quot; under the instructions, then click again.\" style=\"font: 500 14px Roboto, Arial, sans-serif; padding: 7px 16px; border: 0; border-radius: 4px; background: #1a73e8; color: #fff; cursor: pointer;\">Fill in solution</button><span style=\"font: 14px Roboto, Arial, sans-serif; margin-left: 12px;\"></span>\n<script>\nasync function greenaiFill(button, exercises) {\n  const msg = button.nextElementSibling;\n  let ok = false;\n  try {\n    const r = await google.colab.kernel.invokeFunction(\"greenai.fill_in\", [exercises], {});\n    ok = Boolean(r && r.data && r.data[\"application/json\"] && r.data[\"application/json\"].ok);\n  } catch (e) {}\n  msg.textContent = ok ? button.dataset.done : button.dataset.need;\n}\n</script>"))

In [ ]:
# Exercise 4
config_names = list(runs)
for i, pair in enumerate(QA_PAIRS):
    print(f"Q: {pair['question']}")
    for config in config_names:
        r = runs[config]["rows"][i]
        print(f"   {config:28s} {'✓' if r['correct'] else '✗'}  {r['answer']}")
    print()

summary = pd.DataFrame({
    config: {
        "answer accuracy": ___,  # TODO: fraction of correct answers
        "accuracy, Notebook 1's 8 questions": np.mean(
            [r["correct"] for r in run["rows"][:N_SESSION1_QUESTIONS]]),
        "retrieval accuracy": (np.mean([r["retrieval_correct"] for r in run["rows"]])
                               if run["retrieval"] else None),
        "mean prompt tokens": np.mean([r["prompt_tokens"] for r in run["rows"]]),
        "seconds per question": run["elapsed"] / len(QA_PAIRS),
        "g CO2eq per question": ___,  # TODO: grams per question
        "model weights (GB)": load_info[run["model"]]["weights_gb"],
    }
    for config, run in runs.items()
})
summary

In [ ]:
# Check for Exercise 4. Run this cell as is.
for _c, _run in runs.items():
    assert abs(summary[_c]["g CO2eq per question"] - _run["emissions_kg"] * 1000 / len(QA_PAIRS)) < 1e-12
    assert abs(summary[_c]["answer accuracy"] - sum(r["correct"] for r in _run["rows"]) / len(QA_PAIRS)) < 1e-12
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
config_names = list(runs)
for i, pair in enumerate(QA_PAIRS):
    print(f"Q: {pair['question']}")
    for config in config_names:
        r = runs[config]["rows"][i]
        print(f"   {config:28s} {'✓' if r['correct'] else '✗'}  {r['answer']}")
    print()

summary = pd.DataFrame({
    config: {
        "answer accuracy": np.mean([r["correct"] for r in run["rows"]]),
        "accuracy, Notebook 1's 8 questions": np.mean(
            [r["correct"] for r in run["rows"][:N_SESSION1_QUESTIONS]]),
        "retrieval accuracy": (np.mean([r["retrieval_correct"] for r in run["rows"]])
                               if run["retrieval"] else None),
        "mean prompt tokens": np.mean([r["prompt_tokens"] for r in run["rows"]]),
        "seconds per question": run["elapsed"] / len(QA_PAIRS),
        "g CO2eq per question": run["emissions_kg"] * 1000 / len(QA_PAIRS),
        "model weights (GB)": load_info[run["model"]]["weights_gb"],
    }
    for config, run in runs.items()
})
summary
```

</details>

### Save the results and chart them

All three rows go into `all_in_one_leaderboard.csv`, a fresh file separate from the
per-session `leaderboard.csv`. Download it from the Files sidebar to keep it.

In [ ]:
leaderboard = pd.DataFrame([{
    "config": config,
    "model": QUANT_MODEL_NAME if run["model"] == "quantized" else BASE_MODEL_NAME,
    "n_questions": len(QA_PAIRS),
    "duration_sec": round(run["elapsed"], 2),
    "energy_kwh": run["energy_kwh"],
    "emissions_kg_co2eq": run["emissions_kg"],
    **{k.replace(" ", "_").replace(",", "").replace("'", ""): v
       for k, v in summary[config].items()},
} for config, run in runs.items()])
leaderboard.to_csv("all_in_one_leaderboard.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
panels = [("answer accuracy", 100, "accuracy (%)"),
          ("g CO2eq per question", 1, "g CO2eq per question"),
          ("model weights (GB)", 1, "GPU memory for weights (GB)")]
for ax, (row, scale, label) in zip(axes, panels):
    ax.bar(summary.columns, summary.loc[row].astype(float) * scale)
    ax.set_title(label)
    ax.tick_params(axis="x", rotation=20)
    plt.setp(ax.get_xticklabels(), ha="right")
plt.tight_layout()
plt.show()

### ✏️ Exercise 5 — Energy per correct answer (advanced)

Emissions per question treat a wrong answer and a right one alike. Energy per
**correct** answer charges each configuration for the answers it got wrong too: a cheap
configuration that is often wrong can cost more per useful answer than an expensive one.
1 kWh = 3,600,000 J.

Complete `joules` and `joules_per_correct`.

<details><summary><b>Hints</b> (click to open)</summary>

- Multiply kWh by 3.6e6.
- `sum(r["correct"] for r in rows)` counts the correct answers.

</details>

In [ ]:
from IPython.display import HTML, display

display(HTML("<button onclick=\"greenaiFill(this, [5])\" data-done=\"✓ Now run the exercise cell below: the solution replaces its code.\" data-need=\"First run the cell &quot;Run this cell first&quot; under the instructions, then click again.\" style=\"font: 500 14px Roboto, Arial, sans-serif; padding: 7px 16px; border: 0; border-radius: 4px; background: #1a73e8; color: #fff; cursor: pointer;\">Fill in solution</button><span style=\"font: 14px Roboto, Arial, sans-serif; margin-left: 12px;\"></span>\n<script>\nasync function greenaiFill(button, exercises) {\n  const msg = button.nextElementSibling;\n  let ok = false;\n  try {\n    const r = await google.colab.kernel.invokeFunction(\"greenai.fill_in\", [exercises], {});\n    ok = Boolean(r && r.data && r.data[\"application/json\"] && r.data[\"application/json\"].ok);\n  } catch (e) {}\n  msg.textContent = ok ? button.dataset.done : button.dataset.need;\n}\n</script>"))

In [ ]:
# Exercise 5
def joules(kwh):
    # TODO: kWh to joules
    ___


def joules_per_correct(kwh, rows):
    # TODO: energy in joules divided by the number of correct rows (infinity if none)
    ___

rows = []
for config, run in runs.items():
    n_correct = sum(r["correct"] for r in run["rows"])
    rows.append({"config": config,
                 "accuracy": n_correct / len(run["rows"]),
                 "J per question": joules(run["energy_kwh"]) / len(run["rows"]),
                 "J per correct answer": joules_per_correct(run["energy_kwh"], run["rows"])})
efficiency = pd.DataFrame(rows)
efficiency

In [ ]:
# Check for Exercise 5. Run this cell as is.
assert joules(0.001) == 3600.0
assert joules_per_correct(0.001, [{"correct": True}, {"correct": False}]) == 3600.0
assert joules_per_correct(0.001, [{"correct": False}]) == float("inf")
assert len(efficiency) == len(runs)
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
def joules(kwh):
    return kwh * 3.6e6


def joules_per_correct(kwh, rows):
    n_correct = sum(r["correct"] for r in rows)
    return joules(kwh) / n_correct if n_correct else float("inf")

rows = []
for config, run in runs.items():
    n_correct = sum(r["correct"] for r in run["rows"])
    rows.append({"config": config,
                 "accuracy": n_correct / len(run["rows"]),
                 "J per question": joules(run["energy_kwh"]) / len(run["rows"]),
                 "J per correct answer": joules_per_correct(run["energy_kwh"], run["rows"])})
efficiency = pd.DataFrame(rows)
efficiency
```

</details>

### ✏️ Exercise 6 — Which configurations are worth considering? The Pareto front (advanced)

No single number ranks the configurations: one may be cheaper, another more
accurate. A configuration is **dominated** if some other configuration is at least as
cheap **and** at least as accurate, and strictly better in one of the two. The
configurations that nobody dominates form the **Pareto front**: the only sensible choices,
whatever weight you give to cost against accuracy.

Complete `dominates` and `pareto_front`. The chart marks the front in green.

**Reflection:** which configurations are on the front? If one is dominated, what would
have to change for it to join the front?

<details><summary><b>Hints</b> (click to open)</summary>

- `a[cost] <= b[cost] and a[quality] >= b[quality]`: a is no worse.
- Strictly better: `a[cost] < b[cost] or a[quality] > b[quality]`.
- Keep a row if `not any(dominates(other, row, cost, quality) for other in rows)`.

</details>

In [ ]:
from IPython.display import HTML, display

display(HTML("<button onclick=\"greenaiFill(this, [6])\" data-done=\"✓ Now run the exercise cell below: the solution replaces its code.\" data-need=\"First run the cell &quot;Run this cell first&quot; under the instructions, then click again.\" style=\"font: 500 14px Roboto, Arial, sans-serif; padding: 7px 16px; border: 0; border-radius: 4px; background: #1a73e8; color: #fff; cursor: pointer;\">Fill in solution</button><span style=\"font: 14px Roboto, Arial, sans-serif; margin-left: 12px;\"></span>\n<script>\nasync function greenaiFill(button, exercises) {\n  const msg = button.nextElementSibling;\n  let ok = false;\n  try {\n    const r = await google.colab.kernel.invokeFunction(\"greenai.fill_in\", [exercises], {});\n    ok = Boolean(r && r.data && r.data[\"application/json\"] && r.data[\"application/json\"].ok);\n  } catch (e) {}\n  msg.textContent = ok ? button.dataset.done : button.dataset.need;\n}\n</script>"))

In [ ]:
# Exercise 6
def dominates(a, b, cost, quality):
    """True if row a dominates row b: no worse on both, strictly better on at least one."""
    # TODO
    ___


def pareto_front(table, cost, quality):
    rows = [row for _, row in table.iterrows()]
    # TODO: the "config" of every row that no other row dominates
    ___

front = pareto_front(efficiency, cost="J per question", quality="accuracy")
print("On the Pareto front:", front)

fig, ax = plt.subplots(figsize=(6, 4))
for _, row in efficiency.iterrows():
    on_front = row["config"] in front
    ax.scatter(row["J per question"], row["accuracy"] * 100, s=80,
               color="tab:green" if on_front else "tab:gray")
    ax.annotate(row["config"], (row["J per question"], row["accuracy"] * 100),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xlabel("energy per question (J)")
ax.set_ylabel("accuracy (%)")
ax.set_title("Green: no other configuration is both cheaper and better")
plt.tight_layout()
plt.show()

In [ ]:
# Check for Exercise 6. Run this cell as is.
_t = pd.DataFrame({"config": ["a", "b", "c", "d"], "J": [1.0, 2.0, 3.0, 2.0], "acc": [0.5, 0.8, 0.7, 0.8]})
assert dominates(_t.iloc[1], _t.iloc[2], "J", "acc"), "b is cheaper and more accurate than c"
assert not dominates(_t.iloc[1], _t.iloc[3], "J", "acc"), "equal rows do not dominate each other"
assert pareto_front(_t, "J", "acc") == ["a", "b", "d"]
assert set(front) <= set(efficiency["config"]) and len(front) >= 1
print("✓ Exercise 6 passed")

<details><summary><b>Solution to Exercise 6</b> (open it after you have tried)</summary>

```python
def dominates(a, b, cost, quality):
    """True if row a dominates row b: no worse on both, strictly better on at least one."""
    no_worse = a[cost] <= b[cost] and a[quality] >= b[quality]
    better = a[cost] < b[cost] or a[quality] > b[quality]
    return no_worse and better


def pareto_front(table, cost, quality):
    rows = [row for _, row in table.iterrows()]
    return [r["config"] for r in rows
            if not any(dominates(other, r, cost, quality) for other in rows)]

front = pareto_front(efficiency, cost="J per question", quality="accuracy")
print("On the Pareto front:", front)

fig, ax = plt.subplots(figsize=(6, 4))
for _, row in efficiency.iterrows():
    on_front = row["config"] in front
    ax.scatter(row["J per question"], row["accuracy"] * 100, s=80,
               color="tab:green" if on_front else "tab:gray")
    ax.annotate(row["config"], (row["J per question"], row["accuracy"] * 100),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xlabel("energy per question (J)")
ax.set_ylabel("accuracy (%)")
ax.set_title("Green: no other configuration is both cheaper and better")
plt.tight_layout()
plt.show()
```

</details>

**Reflection:** How much did the model know with no documents, in fp16 and quantized? Did
quantization change which questions it got right? What did retrieval add, and what did it
cost in prompt length and emissions? Compare the three panels: which configuration would
you call the most efficient, and what are you measuring efficiency *per*?

## Part 2 — Translation: the same three configurations

The Galician passage from Notebook 01e, translated three ways: by the fp16 model, by the
quantized model, and by the fp16 model with a worked example retrieved from a six-sentence
translation memory (Notebook 3's simplified kNN-MT). Translation has no closed-book
question, so the first two runs are the plain translator prompt.

In [ ]:
TRANSLATION_MEMORY = [
    {"gl": "O meu curmán Fernando aguantaba o trebón que lle estaba a caer por riba.",
     "en": "My cousin Fernando was weathering the storm breaking over his head."},
    {"gl": "Estou rodeado de merda, carallo! Veño dar un discurso do carallo, e o carallo do discurso está na mesa dalgún pringhao papando hostias!",
     "en": "I'm surrounded by morons, damn it! I come here to give a fucking speech, and it is sitting on some loser's desk twiddling his fucking thumbs!"},
    {"gl": "Un paso máis atrás, o alcalde suaba puro medo.",
     "en": "One step behind him, the mayor was sweating pure fear."},
    {"gl": "Pero que dúas horas nin que merda! Ti toleas! Como vou ter dúas horas aí a esa xente papando hostias!",
     "en": "What do you mean, two fucking hours? Are you fucking crazy? How am I supposed to keep all those people there twiddling their fucking thumbs for two hours?"},
    {"gl": "Xuro que fun capaz de ler o pensamento do alcalde naquel momento.",
     "en": "I swear I could read the mayor's thoughts at that moment."},
    {"gl": "Á noitiña, xa máis calmo, pagou os cubatas.",
     "en": "That evening, once he had calmed down, Fernando bought me some rum and cokes."},
]

FULL_TEXT_GL = (
    "Ás veces é mellor estar calado. Unha tía miña, que todo o solucionaba con ditos e "
    "refráns, dicía algo así como \"un é escravo do que di e dono do que cala\". Sempre me "
    "pareceu un sabio aforismo, e se cadra debería terlle feito caso nalgunhas ocasións, "
    "aquelas que, aínda que raras, contribuíron a dar bruscos cambios de sentido á miña vida."
)
REFERENCE_EN = (
    "Sometimes it's better to stay quiet. I had an aunt who solved everything with sayings "
    "and proverbs; she used to say something like, 'You're a slave to what you say, and "
    "master of what you keep to yourself.' It always struck me as a wise saying, and maybe "
    "I should have listened to it on a few occasions — the rare ones that, even so, ended up "
    "sharply changing the course of my life."
)

tm_embeddings = embedder.encode([pair["gl"] for pair in TRANSLATION_MEMORY],
                                normalize_embeddings=True)

def retrieve_translation_example(source_text):
    q_emb = embedder.encode([source_text], normalize_embeddings=True)[0]
    sims = tm_embeddings @ q_emb
    best_idx = int(np.argmax(sims))
    return TRANSLATION_MEMORY[best_idx], sims


best_example, tm_sims = retrieve_translation_example(FULL_TEXT_GL)
for pair, sim in sorted(zip(TRANSLATION_MEMORY, tm_sims), key=lambda x: -x[1]):
    print(f"{sim:.3f}  {pair['gl']}")
print(f"\nRetrieved: {best_example['gl']}\n       -> {best_example['en']}")

### ✏️ Exercise 7 — Three translation configurations

Fill in `MT_CONFIGS`. Each entry is a dict with `"model"` (`"fp16"` or
`"quantized"`) and `"example"` (`True` to add the retrieved translation-memory example to
the prompt).

<details><summary><b>Hints</b> (click to open)</summary>

- Compare with `CONFIGS` in Step 4.

</details>

In [ ]:
from IPython.display import HTML, display

display(HTML("<button onclick=\"greenaiFill(this, [7])\" data-done=\"✓ Now run the exercise cell below: the solution replaces its code.\" data-need=\"First run the cell &quot;Run this cell first&quot; under the instructions, then click again.\" style=\"font: 500 14px Roboto, Arial, sans-serif; padding: 7px 16px; border: 0; border-radius: 4px; background: #1a73e8; color: #fff; cursor: pointer;\">Fill in solution</button><span style=\"font: 14px Roboto, Arial, sans-serif; margin-left: 12px;\"></span>\n<script>\nasync function greenaiFill(button, exercises) {\n  const msg = button.nextElementSibling;\n  let ok = false;\n  try {\n    const r = await google.colab.kernel.invokeFunction(\"greenai.fill_in\", [exercises], {});\n    ok = Boolean(r && r.data && r.data[\"application/json\"] && r.data[\"application/json\"].ok);\n  } catch (e) {}\n  msg.textContent = ok ? button.dataset.done : button.dataset.need;\n}\n</script>"))

In [ ]:
# Exercise 7
def translate(model, text, example=None, max_new_tokens=200):
    system_content = ("You are a professional translator. Translate the user's Galician text "
                      "into fluent English.")
    if example is not None:
        system_content += (" Here is one worked example of the style expected:\n"
                           f"Galician: {example['gl']}\nEnglish: {example['en']}\n"
                           "Output only the translation of the new text.")
    else:
        system_content += " Output only the translation."
    messages = [{"role": "system", "content": system_content}, {"role": "user", "content": text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


MT_CONFIGS = {
    "general-llm": ___,  # TODO: fp16, no example
    "general-llm-quantized": ___,  # TODO: quantized, no example
    "general-llm-retrieval-augmented": ___,  # TODO: fp16, with the retrieved example
}

mt_runs = {}
for name, spec in MT_CONFIGS.items():
    tracker = make_tracker(f"green-3c-mt-{name}")
    tracker.start()
    start_time = time.time()
    example = retrieve_translation_example(FULL_TEXT_GL)[0] if spec["example"] else None
    translation = translate(MODELS[spec["model"]], FULL_TEXT_GL, example=example)
    elapsed = time.time() - start_time
    emissions_kg = tracker.stop()
    mt_runs[name] = {"translation": translation, "elapsed": elapsed,
                     "emissions_kg": emissions_kg, "energy_kwh": energy_kwh_of(tracker)}
    print(f"--- {name} ({elapsed:.2f} s, {emissions_kg * 1000:.4f} g CO2eq) ---\n{translation}\n")

In [ ]:
# Check for Exercise 7. Run this cell as is.
assert MT_CONFIGS == {
    "general-llm": {"model": "fp16", "example": False},
    "general-llm-quantized": {"model": "quantized", "example": False},
    "general-llm-retrieval-augmented": {"model": "fp16", "example": True},
}
assert all(isinstance(r["translation"], str) and r["translation"] for r in mt_runs.values())
print("✓ Exercise 7 passed")

<details><summary><b>Solution to Exercise 7</b> (open it after you have tried)</summary>

```python
def translate(model, text, example=None, max_new_tokens=200):
    system_content = ("You are a professional translator. Translate the user's Galician text "
                      "into fluent English.")
    if example is not None:
        system_content += (" Here is one worked example of the style expected:\n"
                           f"Galician: {example['gl']}\nEnglish: {example['en']}\n"
                           "Output only the translation of the new text.")
    else:
        system_content += " Output only the translation."
    messages = [{"role": "system", "content": system_content}, {"role": "user", "content": text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


MT_CONFIGS = {
    "general-llm": {"model": "fp16", "example": False},
    "general-llm-quantized": {"model": "quantized", "example": False},
    "general-llm-retrieval-augmented": {"model": "fp16", "example": True},
}

mt_runs = {}
for name, spec in MT_CONFIGS.items():
    tracker = make_tracker(f"green-3c-mt-{name}")
    tracker.start()
    start_time = time.time()
    example = retrieve_translation_example(FULL_TEXT_GL)[0] if spec["example"] else None
    translation = translate(MODELS[spec["model"]], FULL_TEXT_GL, example=example)
    elapsed = time.time() - start_time
    emissions_kg = tracker.stop()
    mt_runs[name] = {"translation": translation, "elapsed": elapsed,
                     "emissions_kg": emissions_kg, "energy_kwh": energy_kwh_of(tracker)}
    print(f"--- {name} ({elapsed:.2f} s, {emissions_kg * 1000:.4f} g CO2eq) ---\n{translation}\n")
```

</details>

### Evaluate: four metrics

BLEU, BERTScore, an LLM judge, and an independent judge by hand, as in Notebooks 1–3. Each
metric scores all three translations in one tracked call. The judge is the fp16 model for
all three candidates, so the same judge grades every translation; it is still grading
output from its own model family.

### ✏️ Exercise 8 — BLEU through measure_metric

`measure_metric(metric, score_all)` runs `score_all(CANDIDATES)` under a tracker.
Write the BLEU scorer: a function that takes the candidates dict and returns
`{name: BLEU score against REFERENCE_EN, rounded to 2 decimals}`.

<details><summary><b>Hints</b> (click to open)</summary>

- `lambda cands: {...}` is a one-line function.
- The candidate goes in the first list, the reference in the nested second list.

</details>

In [ ]:
from IPython.display import HTML, display

display(HTML("<button onclick=\"greenaiFill(this, [8])\" data-done=\"✓ Now run the exercise cell below: the solution replaces its code.\" data-need=\"First run the cell &quot;Run this cell first&quot; under the instructions, then click again.\" style=\"font: 500 14px Roboto, Arial, sans-serif; padding: 7px 16px; border: 0; border-radius: 4px; background: #1a73e8; color: #fff; cursor: pointer;\">Fill in solution</button><span style=\"font: 14px Roboto, Arial, sans-serif; margin-left: 12px;\"></span>\n<script>\nasync function greenaiFill(button, exercises) {\n  const msg = button.nextElementSibling;\n  let ok = false;\n  try {\n    const r = await google.colab.kernel.invokeFunction(\"greenai.fill_in\", [exercises], {});\n    ok = Boolean(r && r.data && r.data[\"application/json\"] && r.data[\"application/json\"].ok);\n  } catch (e) {}\n  msg.textContent = ok ? button.dataset.done : button.dataset.need;\n}\n</script>"))

In [ ]:
# Exercise 8
CANDIDATES = {name: run["translation"] for name, run in mt_runs.items()}
eval_runs = {}


def measure_metric(metric, score_all):
    tracker = make_tracker(f"green-3c-eval-{metric}")
    tracker.start()
    start_time = time.time()
    try:
        scores = score_all(CANDIDATES)
    finally:
        elapsed = time.time() - start_time
        emissions_kg = tracker.stop()
    eval_runs[metric] = {"scores": scores, "elapsed": elapsed, "emissions_kg": emissions_kg}
    for name, score in scores.items():
        print(f"{metric:16s} {name:32s} {score}")


measure_metric("bleu", ___)  # TODO: a function from the candidates dict to {name: BLEU}


def bertscore_all(cands):
    from bert_score import score as bertscore
    _, _, f1 = bertscore(list(cands.values()), [REFERENCE_EN] * len(cands),
                         model_type="distilbert-base-uncased", verbose=False)
    return {name: round(v, 4) for name, v in zip(cands, f1.tolist())}


try:
    measure_metric("bertscore", bertscore_all)
except Exception as e:
    print(f"BERTScore failed in this environment ({type(e).__name__}: {e}). Continuing without it.")

JUDGE_INSTRUCTIONS = (
    "You are an expert translation evaluator. Given a source text, a candidate "
    "translation, and a reference translation, rate the candidate's quality from 1 "
    "(very poor) to 10 (excellent), considering both accuracy and fluency. Output "
    "ONLY the number."
)


def judge_request(candidate):
    return (f"Source (Galician): {FULL_TEXT_GL}\n\n"
            f"Reference translation: {REFERENCE_EN}\n\n"
            f"Candidate translation: {candidate}\n\n"
            "Score (1-10):")


def qwen_judge(candidate):
    messages = [{"role": "system", "content": JUDGE_INSTRUCTIONS},
                {"role": "user", "content": judge_request(candidate)}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model_fp16.device)
    with torch.no_grad():
        output = model_fp16.generate(**inputs, max_new_tokens=10, do_sample=False)
    return parse_score(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:],
                                        skip_special_tokens=True))


measure_metric("llm_judge_qwen", lambda cands: {n: qwen_judge(t) for n, t in cands.items()})

In [ ]:
# Check for Exercise 8. Run this cell as is.
_b = eval_runs["bleu"]["scores"]
assert set(_b) == set(CANDIDATES)
_n = "general-llm"
assert _b[_n] == round(sacrebleu.corpus_bleu([CANDIDATES[_n]], [[REFERENCE_EN]]).score, 2), \
    "the candidate goes first, the reference second"
print("✓ Exercise 8 passed")

<details><summary><b>Solution to Exercise 8</b> (open it after you have tried)</summary>

```python
CANDIDATES = {name: run["translation"] for name, run in mt_runs.items()}
eval_runs = {}


def measure_metric(metric, score_all):
    tracker = make_tracker(f"green-3c-eval-{metric}")
    tracker.start()
    start_time = time.time()
    try:
        scores = score_all(CANDIDATES)
    finally:
        elapsed = time.time() - start_time
        emissions_kg = tracker.stop()
    eval_runs[metric] = {"scores": scores, "elapsed": elapsed, "emissions_kg": emissions_kg}
    for name, score in scores.items():
        print(f"{metric:16s} {name:32s} {score}")


measure_metric("bleu", lambda cands: {
    name: round(sacrebleu.corpus_bleu([text], [[REFERENCE_EN]]).score, 2)
    for name, text in cands.items()})


def bertscore_all(cands):
    from bert_score import score as bertscore
    _, _, f1 = bertscore(list(cands.values()), [REFERENCE_EN] * len(cands),
                         model_type="distilbert-base-uncased", verbose=False)
    return {name: round(v, 4) for name, v in zip(cands, f1.tolist())}


try:
    measure_metric("bertscore", bertscore_all)
except Exception as e:
    print(f"BERTScore failed in this environment ({type(e).__name__}: {e}). Continuing without it.")

JUDGE_INSTRUCTIONS = (
    "You are an expert translation evaluator. Given a source text, a candidate "
    "translation, and a reference translation, rate the candidate's quality from 1 "
    "(very poor) to 10 (excellent), considering both accuracy and fluency. Output "
    "ONLY the number."
)


def judge_request(candidate):
    return (f"Source (Galician): {FULL_TEXT_GL}\n\n"
            f"Reference translation: {REFERENCE_EN}\n\n"
            f"Candidate translation: {candidate}\n\n"
            "Score (1-10):")


def qwen_judge(candidate):
    messages = [{"role": "system", "content": JUDGE_INSTRUCTIONS},
                {"role": "user", "content": judge_request(candidate)}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model_fp16.device)
    with torch.no_grad():
        output = model_fp16.generate(**inputs, max_new_tokens=10, do_sample=False)
    return parse_score(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:],
                                        skip_special_tokens=True))


measure_metric("llm_judge_qwen", lambda cands: {n: qwen_judge(t) for n, t in cands.items()})
```

</details>

#### The independent judge: ChatGPT, by hand

Paste each prompt below into a **new** chat in ChatGPT (or Claude, or Gemini), so each
score is given without the others in view, then fill the numbers in the next cell. Scores
left as `None` are skipped.

In [ ]:
for name, text in CANDIDATES.items():
    print(f"===== prompt for: {name} =====")
    print(JUDGE_INSTRUCTIONS + "\n\n" + judge_request(text) + "\n")

In [ ]:
CHATGPT_JUDGE_SCORES = {
    "general-llm": None,
    "general-llm-quantized": None,
    "general-llm-retrieval-augmented": None,
}

eval_runs["llm_judge_chatgpt"] = {
    "scores": {n: float(s) for n, s in CHATGPT_JUDGE_SCORES.items() if s is not None},
    "elapsed": None, "emissions_kg": None,
}

### Translation results in one table

Cost of producing each translation, and every metric's score for it. Saved as
`all_in_one_mt_results.csv`.

In [ ]:
mt_table = pd.DataFrame({
    name: {
        "translation g CO2eq": run["emissions_kg"] * 1000,
        "translation seconds": run["elapsed"],
        **{metric: ev["scores"].get(name) for metric, ev in eval_runs.items()},
    }
    for name, run in mt_runs.items()
})
mt_table.T.assign(translation=[run["translation"] for run in mt_runs.values()]).to_csv(
    "all_in_one_mt_results.csv")

metrics = [m for m in eval_runs if eval_runs[m]["scores"]]
fig, axes = plt.subplots(1, len(metrics), figsize=(3.4 * len(metrics), 3.5), squeeze=False)
labels = ["fp16", "quantized", "fp16 +\nretrieval"]
for ax, metric in zip(axes[0], metrics):
    ax.bar(labels, [eval_runs[metric]["scores"].get(n, np.nan) for n in CANDIDATES])
    ax.set_title(metric)
plt.tight_layout()
plt.show()
mt_table

**Reflection:** Read the three translations side by side. Where does the quantized
model's wording differ from fp16's? Did the retrieved example change anything around
"Unha tía miña"? Do the four metrics agree on the ranking, and which would you trust for a
single passage?

## Wrap-up: every tracked run in this notebook

g CO2eq on a log scale: building the index, three quiz runs, three translations and the
evaluation metrics.

In [ ]:
wrapup = [("Index 26 documents (once)", emissions_index_kg)]
wrapup += [(f"QA: {c}", r["emissions_kg"]) for c, r in runs.items()]
wrapup += [(f"MT: {n}", r["emissions_kg"]) for n, r in mt_runs.items()]
wrapup += [(f"Eval: {m}", r["emissions_kg"]) for m, r in eval_runs.items() if r["emissions_kg"]]
wrapup = pd.DataFrame(wrapup, columns=["run", "kg"]).assign(g_co2eq=lambda d: d.kg * 1000)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(wrapup["run"], wrapup["g_co2eq"])
ax.set_xscale("log")
ax.set_xlabel("g CO2eq (log scale)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Key takeaways**

- Closed book, the model answers few of the quiz questions, and many wrong answers are
  confident and specific. Everything above that line comes from documents in the prompt.
- Quantization changes what the model remembers as well as what it costs. On a T4 with the
  GPTQ kernel that loads here, the saving is memory, not time or energy per question.
- Retrieval buys most of the accuracy for a few hundred prompt tokens per question. Its
  failures are misses by the retriever, which Notebook 3 traced to encoder truncation,
  acronyms and near-duplicate documents.
- Running all three configurations in one session, on the same hardware, removes the
  session-to-session noise in the separate leaderboards (different GPUs, different grid
  intensity, different days).

## Share-out (discussion)

- Which questions did the closed-book fp16 model get right? Did the quantized model get
  the same ones?
- Pick a question the closed-book model got wrong with a confident, specific answer. How
  would a user of this system notice?
- Rank the three configurations by accuracy, by g CO2eq per question, and by GPU memory.
  Are the rankings the same? Which one would you deploy, and for what task?
- How do today's numbers compare with the separate Session 1–3 leaderboards? What could
  explain differences, given the same model and questions?

## Code literacy check

Before you're done, pick two or three code cells above that you ran and read today, and
add `#` comments explaining in your own words what each meaningful line or block does and
*why* it's there, so that a classmate who hasn't seen the code could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini, whichever you have
   access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, check the code again: who's
   right?